In [ ]:
# Install deps
!pip install -q pandas scikit-learn joblib tqdm

In [ ]:
import os, zipfile, requests, sys
url = "https://github.com/vedangvatsa/ai-detection-at-scale/archive/refs/heads/main.zip"
r = requests.get(url)
open("repo.zip", "wb").write(r.content)
with zipfile.ZipFile("repo.zip") as z:
    z.extractall(".")
os.rename("ai-detection-at-scale-main", "ai-detection-at-scale")
os.chdir("ai-detection-at-scale")
print("Repo ready")

In [ ]:
# Download data assets
!python scripts/download_assets.py

In [ ]:
# Run 35-feature evaluation (inline)
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from tool.feature_extractor import extract_features, ORIGINAL_FEATURE_COLS, EXTENDED_FEATURE_COLS

SAMPLE_SIZE = 10000
RANDOM_SEED = 42

print("Loading raw corpus (small sample)...")
df = pd.read_parquet("data/corpus_raw.parquet", columns=['text', 'label', 'register'])
df = df.sample(n=min(SAMPLE_SIZE, len(df)), random_state=RANDOM_SEED)
print(f"Sampled {len(df)} texts")

print("Extracting 35 features...")
features_11 = []
features_35 = []
labels = []

for _, row in df.iterrows():
    text = row['text']
    if not isinstance(text, str) or len(text.strip()) < 20:
        continue
    
    feats = extract_features(text, extended=True)
    if feats is None:
        continue
    
    # 11 features
    feats_11 = {k: feats[k] for k in ORIGINAL_FEATURE_COLS if k in feats}
    features_11.append(feats_11)
    
    # 35 features
    feats_35 = {k: feats[k] for k in ORIGINAL_FEATURE_COLS + EXTENDED_FEATURE_COLS if k in feats}
    features_35.append(feats_35)
    
    labels.append(row['label'])

print(f"Valid extractions: {len(labels)}")

# Convert to DataFrames
df_11 = pd.DataFrame(features_11)
df_35 = pd.DataFrame(features_35)
y = np.array(labels)

# Train/test split
X_train_11, X_test_11, y_train, y_test = train_test_split(df_11, y, test_size=0.2, random_state=RANDOM_SEED)
X_train_35, X_test_35, _, _ = train_test_split(df_35, y, test_size=0.2, random_state=RANDOM_SEED)

# Train 11-feature model
print("Training 11-feature model...")
rf_11 = RandomForestClassifier(n_estimators=200, random_state=RANDOM_SEED, n_jobs=-1)
rf_11.fit(X_train_11, y_train)
auc_11 = roc_auc_score(y_test, rf_11.predict_proba(X_test_11)[:, 1])
print(f"11-feature AUC: {auc_11:.4f}")

# Train 35-feature model
print("Training 35-feature model...")
rf_35 = RandomForestClassifier(n_estimators=200, random_state=RANDOM_SEED, n_jobs=-1)
rf_35.fit(X_train_35, y_train)
auc_35 = roc_auc_score(y_test, rf_35.predict_proba(X_test_35)[:, 1])
print(f"35-feature AUC: {auc_35:.4f}")

# Save results
results = pd.DataFrame([{
    'n_features_11': len(ORIGINAL_FEATURE_COLS),
    'n_features_35': len(ORIGINAL_FEATURE_COLS) + len(EXTENDED_FEATURE_COLS),
    'auc_11': auc_11,
    'auc_35': auc_35,
    'improvement': auc_35 - auc_11,
}])
results.to_csv('/kaggle/working/feature_35_comparison.csv', index=False)
print("Results saved")

In [ ]:
# Copy results to output
import shutil
shutil.copy("results/feature_35_comparison.csv", "/kaggle/working/")
print("Output ready")